# Proyecto 6: Evaluación de retrieval

**Estudiante:** Luis Baca Sandoval

**Cuaderno base:** `Cuaderno23-CC0C2.ipynb` 

**Tema central:** métricas de evaluación para sistemas de recuperación de información
(`precision@k`, `recall@k`, `MRR` y `nDCG`), comparando al menos dos configuraciones de retrieval.

---

### Autoría del código

Cada celda de código está etiquetada en su primera línea con una de estas marcas:

- `# [REUTILIZADO de Cuaderno23]` — código tomado tal cual del cuaderno base.
- `# [MODIFICADO de Cuaderno23]` — código del cuaderno base que adapté/corregí.
- `# [AGREGADO por el estudiante]` — código nuevo escrito por mí para este proyecto.
- `# [APOYO DE IA]` — código donde usé IA como apoyo.

> **Aportes propios principales:** (1) implementar **nDCG** (que el Cuaderno23 **no** trae),
> (2) **corregir** `recall@k` para que mida cobertura real, (3) **enriquecer** el corpus con
> distractores y reescribir el benchmark con **consultas parafraseadas** (con desajuste de
> vocabulario), y (4) construir un *harness* de evaluación centrado en retrieval que compara una
> **línea base (BM25)** contra una **variante (BM25 + expansión de consulta)** y evaluando diferentes valores de top_k.


## CELDA DE VERIFICACIÓN PERSONAL

In [1]:
import random

STUDENT_NAME   = "Luis Baca Sandoval"
EXECUTION_DATE = "2026-06-22"
BASE_NOTEBOOK  = "Cuaderno23-CC0C2.ipynb"
MODEL_NAME     = "Retrieval determinista en stdlib: BM25 simplificado + denso TF-IDF (sin GPU, offline)"
SEED           = 42
QUERIES        = 6  # número de consultas del benchmark de evaluación
VARIANT        = ("Agrego nDCG@k, corrijo recall@k y comparo BM25 (línea base) "
                  "vs BM25 + expansión de consulta; además evalúo diferentes valores de top_k.")

random.seed(SEED)

print("Estudiante:", STUDENT_NAME)
print("Fecha:", EXECUTION_DATE)
print("Cuaderno base:", BASE_NOTEBOOK)
print("Modelo/representación:", MODEL_NAME)
print("Semilla:", SEED)
print("Número de consultas:", QUERIES)
print("Variante:", VARIANT)


Estudiante: Luis Baca Sandoval
Fecha: 2026-06-22
Cuaderno base: Cuaderno23-CC0C2.ipynb
Modelo/representación: Retrieval determinista en stdlib: BM25 simplificado + denso TF-IDF (sin GPU, offline)
Semilla: 42
Número de consultas: 6
Variante: Agrego nDCG@k, corrijo recall@k y comparo BM25 (línea base) vs BM25 + expansión de consulta; además evalúo diferentes valores de top_k.


## 1. Objetivo

Construir un **dataset de evaluación** con consultas y documentos relevantes conocidos
(*gold documents*), implementar las métricas **precision@k**, **recall@k**, **MRR** y **nDCG**, y
**comparar dos configuraciones de retrieval** usando esas métricas.

¿Por qué importa? Un sistema RAG solo puede generar respuestas fundamentadas si el paso de
**recuperación** coloca los documentos correctos en las primeras posiciones. Si el retrieval
falla, la generación hereda el error ("garbage in, garbage out"). Evaluar el retrieval de forma
cuantitativa permite **elegir configuraciones por evidencia** y no por intuición.

**Modificación obligatoria del Proyecto 6:** *"cambiar el tamaño de `top_k` y analizar cómo
evolucionan precisión y recall, o comparar dos modelos de embeddings"*. Como el notebook es 100%
offline/stdlib (sin descargar modelos), sustituyo "dos modelos de embeddings" por **dos
configuraciones de retrieval** —lo que el propio enunciado del proyecto pide: *"comparar al menos
dos configuraciones de retrieval usando estas métricas"*— y **además** hago el barrido de `top_k`.
Así cubro ambas variantes (Ejercicio A y Ejercicio B).


## 2. Marco teórico: métricas de recuperación

Sea una consulta $q$ con un conjunto de documentos relevantes (*gold*) $G_q$, y sea
$R_q = (d_1, d_2, \dots)$ la **lista ordenada** que devuelve el retriever.

**Precision@k** — fracción de los $k$ primeros recuperados que es relevante:
$$\text{P@}k = \frac{|\{d_1,\dots,d_k\} \cap G_q|}{k}$$
Penaliza traer ruido en el top-$k$.

**Recall@k** — fracción de los relevantes recuperados dentro del top-$k$:
$$\text{R@}k = \frac{|\{d_1,\dots,d_k\} \cap G_q|}{|G_q|}$$
Penaliza dejar relevantes fuera.

> **Tensión precisión–recall:** al **aumentar** `k`, el recall **nunca baja** (caben más
> relevantes) pero la precisión **tiende a bajar** (entra más ruido). El `k` óptimo equilibra ambos.

**MRR (Mean Reciprocal Rank)** — premia colocar el **primer** relevante lo más arriba posible. Si
el primer relevante aparece en la posición $\text{rank}$:
$$\text{RR} = \frac{1}{\text{rank}}, \qquad \text{MRR} = \frac{1}{|Q|}\sum_{q\in Q}\text{RR}_q$$
Limitación: solo mira el **primer** acierto; ignora el resto cuando hay varios relevantes.

**nDCG@k (normalized Discounted Cumulative Gain)** — única métrica aquí **sensible al orden
completo**. Con relevancia binaria $rel_i \in \{0,1\}$:
$$\text{DCG@}k = \sum_{i=1}^{k} \frac{rel_i}{\log_2(i+1)}, \qquad
  \text{IDCG@}k = \sum_{i=1}^{\min(k,|G_q|)} \frac{1}{\log_2(i+1)}, \qquad
  \text{nDCG@}k = \frac{\text{DCG@}k}{\text{IDCG@}k}$$
El descuento $1/\log_2(i+1)$ hace que un acierto en la posición 1 valga más que en la 5; el IDCG
(ranking ideal) normaliza a $[0,1]$, por lo que nDCG es **comparable entre consultas**.


## 3. Configuración e imports

In [2]:
# [REUTILIZADO de Cuaderno23] 
from __future__ import annotations

from dataclasses import dataclass, field
from typing import Any, Dict, List, Callable
from collections import Counter, defaultdict
import math
import re

## 4. Corpus controlado y benchmark de evaluación

Partimos del corpus de 8 documentos del Cuaderno23 y agrego 4 distractores
(`doc_vectordb_01`, `doc_chunking_01`, `doc_prompt_01`, `doc_eval_01`): documentos temáticamente
cercanos que **comparten vocabulario** con las consultas pero **no son relevantes**. Sin
distractores, con tan pocos documentos, cualquier `k` recupera todo y las métricas se saturan
(recall triv/ = 1); los distractores hacen el problema discriminativo.

También **reescribo el benchmark** con **consultas parafraseadas**: preguntas que evitan repetir
los términos exactos de los documentos relevantes (*desajuste de vocabulario*, el caso realista de
la búsqueda). Dos consultas tienen **dos** documentos relevantes, para que `recall` y `nDCG`
aporten señal que `MRR` por sí solo no captura.

In [3]:
# [MODIFICADO de Cuaderno23] (corpus base + 4 distractores)
@dataclass
class Document:
    id: str
    title: str
    text: str
    metadata: Dict[str, Any] = field(default_factory=dict)


documents = [
    # --- 8 documentos base (REUTILIZADOS del Cuaderno23) ---
    Document("doc_rag_01", "RAG y evidencia externa",
        "RAG conecta un LLM con documentos externos. Primero recupera evidencia relevante y luego "
        "genera una respuesta fundamentada. Esto reduce respuestas sin soporte documental.",
        {"tema": "rag", "rol": "base"}),
    Document("doc_embeddings_01", "Embeddings y búsqueda semántica",
        "Un embedding representa el significado de un texto como vector denso. La búsqueda semántica "
        "recupera documentos por similitud conceptual, aunque no compartan exactamente las mismas palabras.",
        {"tema": "embeddings", "rol": "base"}),
    Document("doc_hybrid_01", "Búsqueda híbrida",
        "BM25 recupera coincidencias exactas de términos y entidades. Los embeddings recuperan "
        "similitud semántica. En sistemas reales, BM25 y embeddings son complementarios.",
        {"tema": "hybrid_search", "rol": "base"}),
    Document("doc_rerank_01", "Reranking y contexto",
        "El reranking reordena los documentos recuperados usando un criterio más preciso. Ayuda a "
        "colocar la evidencia más útil en las primeras posiciones. La compresión contextual reduce ruido.",
        {"tema": "reranking", "rol": "base"}),
    Document("doc_guardrails_01", "Guardrails y verificación",
        "Los guardrails controlan entradas, herramientas, permisos, formato y salidas. La verificación "
        "comprueba si una respuesta está apoyada por evidencia, si respeta políticas y si evita "
        "información no fundamentada.",
        {"tema": "guardrails", "rol": "base"}),
    Document("doc_agents_01", "Agentes con herramientas",
        "Un agente combina LLM, herramientas, memoria, planificación y verificación. El agente decide "
        "cuándo usar una herramienta, observa el resultado y responde cuando la evidencia es suficiente.",
        {"tema": "agentes", "rol": "base"}),
    Document("doc_react_01", "ReAct",
        "ReAct organiza el trabajo del agente en ciclos de pensamiento operativo, acción y observación. "
        "Un ciclo puede buscar evidencia y otro puede calcular o verificar el resultado.",
        {"tema": "react", "rol": "base"}),
    Document("doc_rlhf_01", "RLHF y preferencias",
        "RLHF usa preferencias humanas para alinear modelos. Un reward model aprende a puntuar "
        "respuestas preferidas sobre respuestas rechazadas. Luego la política puede ajustarse con PPO.",
        {"tema": "rlhf", "rol": "base"}),
    # --- 4 distractores (Comparten vocabulario, NO son gold) ---
    Document("doc_vectordb_01", "Bases de datos vectoriales",
        "Una base de datos vectorial almacena vectores densos e índices para búsqueda aproximada de "
        "vecinos cercanos. Permite recuperar documentos por similitud a gran escala con baja latencia.",
        {"tema": "vectordb", "rol": "distractor"}),
    Document("doc_chunking_01", "Chunking de documentos",
        "El chunking divide un documento largo en fragmentos. El tamaño del fragmento afecta la "
        "granularidad de la recuperación y el contexto disponible para el modelo.",
        {"tema": "chunking", "rol": "distractor"}),
    Document("doc_prompt_01", "Ingeniería de prompts",
        "Un prompt bien diseñado guía al modelo de lenguaje. Incluir instrucciones claras y ejemplos "
        "mejora la calidad de la respuesta generada por el LLM.",
        {"tema": "prompting", "rol": "distractor"}),
    Document("doc_eval_01", "Evaluación de sistemas",
        "La evaluación mide la calidad de un sistema con métricas como precisión y recall. Comparar "
        "configuraciones requiere mantener constantes el corpus y las consultas.",
        {"tema": "evaluacion", "rol": "distractor"}),
]


# Benchmark con consultas PARAFRASEADAS (desajuste de vocabulario respecto a los gold docs).
benchmark = [
    {"question": "cómo se conecta un modelo de lenguaje con conocimiento externo para responder con respaldo",
     "gold_docs": {"doc_rag_01"}},
    {"question": "qué método representa el sentido de un texto como vector para comparar conceptos parecidos",
     "gold_docs": {"doc_embeddings_01"}},
    {"question": "cuándo conviene combinar coincidencia exacta de palabras con parecido conceptual",
     "gold_docs": {"doc_hybrid_01", "doc_embeddings_01"}},
    {"question": "cómo reordenar resultados para poner lo más útil al inicio",
     "gold_docs": {"doc_rerank_01"}},
    {"question": "qué mecanismos controlan permisos, formato y salidas de un sistema generativo",
     "gold_docs": {"doc_guardrails_01"}},
    {"question": "qué arquitectura permite a un programa decidir usar acciones y observar resultados",
     "gold_docs": {"doc_agents_01", "doc_react_01"}},
]

print("Documentos en el corpus:", len(documents),
      "(base:", sum(1 for d in documents if d.metadata["rol"] == "base"),
      "| distractores:", sum(1 for d in documents if d.metadata["rol"] == "distractor"), ")")
print("Consultas en el benchmark:", len(benchmark))
print("Consultas con 2 documentos relevantes:",
      sum(1 for c in benchmark if len(c["gold_docs"]) > 1))


Documentos en el corpus: 12 (base: 8 | distractores: 4 )
Consultas en el benchmark: 6
Consultas con 2 documentos relevantes: 2


## 5. Normalización y representación de texto

Reutilizo el pipeline léxico del Cuaderno23: normalización, tokenización, vocabulario e **IDF
suavizado**. El IDF $\big(\log\frac{1+N}{1+df_t}+1\big)$ pondera los términos raros por encima de
los comunes, base tanto de BM25 como del vector TF-IDF denso.

In [4]:
# [REUTILIZADO de Cuaderno23] (celda "Normalización y representación de texto")
def normalizeText(text: str) -> str:
    text = text.lower()
    text = re.sub(r"[^a-záéíóúüñ0-9\s]", " ", text)
    return " ".join(text.split())


def tokenizeText(text: str) -> List[str]:
    return normalizeText(text).split()


def termFrequency(tokens: List[str]) -> Counter:
    return Counter(tokens)


def buildVocabulary(docs: List[Document]) -> List[str]:
    vocab = set()
    for doc in docs:
        vocab.update(tokenizeText(doc.text))
    return sorted(vocab)


def computeIdf(docs: List[Document], vocabulary: List[str]) -> Dict[str, float]:
    n_docs = len(docs)
    doc_freq = defaultdict(int)
    for doc in docs:
        for token in set(tokenizeText(doc.text)):
            doc_freq[token] += 1
    return {term: math.log((1 + n_docs) / (1 + doc_freq[term])) + 1 for term in vocabulary}


vocabulary = buildVocabulary(documents)
idf_values = computeIdf(documents, vocabulary)

print("Tamaño del vocabulario:", len(vocabulary))
print("Primeros términos:", vocabulary[:15])
print("IDF de 'rag':", round(idf_values.get("rag", 0.0), 4),
      "| IDF de 'documentos':", round(idf_values.get("documentos", 0.0), 4))


Tamaño del vocabulario: 193
Primeros términos: ['a', 'acción', 'afecta', 'agente', 'ajustarse', 'al', 'alinear', 'almacena', 'apoyada', 'aprende', 'aproximada', 'aunque', 'ayuda', 'baja', 'base']
IDF de 'rag': 2.8718 | IDF de 'documentos': 1.9555


## 6. Tabla de dimensiones y estructuras de datos

Documentamos las **dimensiones** de las estructuras clave. Cada documento se
representa como un vector TF-IDF cuya dimensión es el tamaño del vocabulario.

In [5]:
# [AGREGADO por el estudiante]
def show_table(headers, rows):
    widths = [max(len(str(h)), *(len(str(r[i])) for r in rows)) for i, h in enumerate(headers)]
    line = " | ".join(str(h).ljust(widths[i]) for i, h in enumerate(headers))
    print(line)
    print("-" * len(line))
    for r in rows:
        print(" | ".join(str(c).ljust(widths[i]) for i, c in enumerate(r)))


dim_rows = [
    ["Documentos del corpus (N)", len(documents)],
    ["  de los cuales distractores", sum(1 for d in documents if d.metadata["rol"] == "distractor")],
    ["Consultas del benchmark (|Q|)", len(benchmark)],
    ["Tamaño del vocabulario (V)", len(vocabulary)],
    ["Dimensión de cada vector TF-IDF", len(vocabulary)],
    ["Forma de la matriz documento-término", f"{len(documents)} x {len(vocabulary)}"],
    ["Total de documentos relevantes (gold)", sum(len(c["gold_docs"]) for c in benchmark)],
]
show_table(["Estructura", "Dimensión / valor"], dim_rows)


Estructura                            | Dimensión / valor
---------------------------------------------------------
Documentos del corpus (N)             | 12               
  de los cuales distractores          | 4                
Consultas del benchmark (|Q|)         | 6                
Tamaño del vocabulario (V)            | 193              
Dimensión de cada vector TF-IDF       | 193              
Forma de la matriz documento-término  | 12 x 193         
Total de documentos relevantes (gold) | 8                


## 7. Recuperadores: BM25, denso (TF-IDF) e híbrido

Reutilizo los tres recuperadores del Cuaderno23:

- **BM25** (esparso/léxico): puntúa coincidencias exactas de términos con saturación de TF y
  normalización por longitud.
- **Denso TF-IDF** (semántico *simulado*): similitud **coseno** entre vectores TF-IDF.
- **Híbrido**: fusión convexa de ambos puntajes normalizados
  ($\text{score} = \alpha\cdot\text{denso} + (1-\alpha)\cdot\text{BM25}$).

> **Limitación:** el "denso" es TF-IDF, que sigue siendo **léxico** (bolsa de palabras). No
> captura sinónimos reales; por eso, frente a consultas parafraseadas, se comporta de forma
> parecida a BM25. Esto será clave para interpretar los resultados.

In [6]:
# [REUTILIZADO de Cuaderno23] (celda "Recuperador léxico tipo BM25 simplificado")
def bm25Score(query: str, doc: Document, docs: List[Document], k1: float = 1.5, b: float = 0.75) -> float:
    query_terms = tokenizeText(query)
    doc_terms = tokenizeText(doc.text)
    doc_tf = termFrequency(doc_terms)
    avg_doc_len = sum(len(tokenizeText(d.text)) for d in docs) / max(len(docs), 1)
    doc_len = len(doc_terms)
    score = 0.0
    for term in query_terms:
        if term not in idf_values:
            continue
        tf = doc_tf[term]
        numerator = tf * (k1 + 1)
        denominator = tf + k1 * (1 - b + b * doc_len / max(avg_doc_len, 1))
        score += idf_values[term] * numerator / max(denominator, 1e-9)
    return score


def bm25Retrieve(query: str, docs: List[Document], k: int = 3) -> List[Dict[str, Any]]:
    scored = [{"id": d.id, "title": d.title, "text": d.text,
               "score": bm25Score(query, d, docs), "retriever": "bm25"} for d in docs]
    scored.sort(key=lambda item: item["score"], reverse=True)
    return scored[:k]


In [7]:
# [REUTILIZADO de Cuaderno23] (celda "Recuperador denso simulado con TF-IDF")
def vectorizeText(text: str, vocabulary: List[str], idf: Dict[str, float]) -> List[float]:
    tokens = tokenizeText(text)
    tf = termFrequency(tokens)
    total = max(len(tokens), 1)
    return [(tf[term] / total) * idf.get(term, 0.0) for term in vocabulary]


def cosineSimilarity(vec_a: List[float], vec_b: List[float]) -> float:
    dot = sum(a * b for a, b in zip(vec_a, vec_b))
    norm_a = math.sqrt(sum(a * a for a in vec_a))
    norm_b = math.sqrt(sum(b * b for b in vec_b))
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return dot / (norm_a * norm_b)


document_vectors = {d.id: vectorizeText(d.text, vocabulary, idf_values) for d in documents}


def denseRetrieve(query: str, docs: List[Document], k: int = 3) -> List[Dict[str, Any]]:
    query_vector = vectorizeText(query, vocabulary, idf_values)
    scored = [{"id": d.id, "title": d.title, "text": d.text,
               "score": cosineSimilarity(query_vector, document_vectors[d.id]),
               "retriever": "dense_tfidf"} for d in docs]
    scored.sort(key=lambda item: item["score"], reverse=True)
    return scored[:k]


In [8]:
# [REUTILIZADO de Cuaderno23] (celda "Búsqueda híbrida: BM25 + recuperación densa")
def minMaxNormalize(scores: Dict[str, float]) -> Dict[str, float]:
    if not scores:
        return {}
    values = list(scores.values())
    lo, hi = min(values), max(values)
    if hi == lo:
        return {k: 0.0 for k in scores}
    return {k: (v - lo) / (hi - lo) for k, v in scores.items()}


def hybridRetrieve(query: str, docs: List[Document], k: int = 3, alpha: float = 0.6) -> List[Dict[str, Any]]:
    bm25_all = bm25Retrieve(query, docs, k=len(docs))
    dense_all = denseRetrieve(query, docs, k=len(docs))
    bm25_norm = minMaxNormalize({i["id"]: i["score"] for i in bm25_all})
    dense_norm = minMaxNormalize({i["id"]: i["score"] for i in dense_all})
    by_id = {d.id: d for d in docs}
    fused = []
    for doc_id, d in by_id.items():
        score = alpha * dense_norm.get(doc_id, 0.0) + (1 - alpha) * bm25_norm.get(doc_id, 0.0)
        fused.append({"id": d.id, "title": d.title, "text": d.text,
                      "score": score, "retriever": "hybrid"})
    fused.sort(key=lambda item: item["score"], reverse=True)
    return fused[:k]


# Inspección rápida del ranking para una consulta parafraseada con 2 gold docs.
demo_q = benchmark[2]["question"]
print("Consulta:", demo_q)
print("Gold:", benchmark[2]["gold_docs"])
for name, fn in [("BM25", bm25Retrieve), ("Denso", denseRetrieve), ("Híbrido", hybridRetrieve)]:
    top = fn(demo_q, documents, k=3)
    print(f"  {name:8s}:", [(t['id'], round(t['score'], 3)) for t in top])


Consulta: cuándo conviene combinar coincidencia exacta de palabras con parecido conceptual
Gold: {'doc_embeddings_01', 'doc_hybrid_01'}
  BM25    : [('doc_embeddings_01', 7.134), ('doc_vectordb_01', 3.972), ('doc_eval_01', 3.519)]
  Denso   : [('doc_embeddings_01', 0.268), ('doc_agents_01', 0.113), ('doc_vectordb_01', 0.109)]
  Híbrido : [('doc_embeddings_01', 1.0), ('doc_vectordb_01', 0.467), ('doc_agents_01', 0.412)]


## 8. Métricas base 

Reutilizo `precisionAtK` y `reciprocalRank` (MRR) del Cuaderno23. Pero su `recallAtK` **no es un
recall real**: devuelve `1.0` si aparece *cualquier* documento relevante en el top-$k$ (es una
tasa de acierto binaria, *hit-rate*). Cuando una consulta tiene **dos** relevantes, recuperar solo
uno **no** debería contar como recall completo.

Por eso **modifico** la función para el recall verdadero
$\text{R@}k = |G_q \cap \text{top}_k| / |G_q|$. Dejo ambas versiones para contrastarlas.

In [9]:
# [REUTILIZADO de Cuaderno23] (celda "Métricas de recuperación")
def precisionAtK(gold_docs: set, retrieved_docs: List[Dict[str, Any]], k: int) -> float:
    retrieved_ids = [item["id"] for item in retrieved_docs[:k]]
    if not retrieved_ids:
        return 0.0
    hits = sum(1 for doc_id in retrieved_ids if doc_id in gold_docs)
    return hits / len(retrieved_ids)


def reciprocalRank(gold_docs: set, retrieved_docs: List[Dict[str, Any]], k: int) -> float:
    for rank, item in enumerate(retrieved_docs[:k], start=1):
        if item["id"] in gold_docs:
            return 1.0 / rank
    return 0.0


def recallAtK_binary(gold_docs: set, retrieved_docs: List[Dict[str, Any]], k: int) -> float:
    # Versión ORIGINAL del Cuaderno23: hit-rate (1.0 si hay AL MENOS un relevante).
    retrieved_ids = {item["id"] for item in retrieved_docs[:k]}
    return 1.0 if gold_docs & retrieved_ids else 0.0


In [10]:
# [MODIFICADO de Cuaderno23] recall real = |gold ∩ top_k| / |gold|
def recallAtK(gold_docs: set, retrieved_docs: List[Dict[str, Any]], k: int) -> float:
    if not gold_docs:
        return 0.0
    retrieved_ids = {item["id"] for item in retrieved_docs[:k]}
    return len(gold_docs & retrieved_ids) / len(gold_docs)


# Demostración de la diferencia en una consulta con 2 documentos relevantes.
caso = benchmark[2]  # gold = {doc_hybrid_01, doc_embeddings_01}
ranked = bm25Retrieve(caso["question"], documents, k=len(documents))
print("Pregunta:", caso["question"])
print("Gold docs:", caso["gold_docs"])
print("Top-2 BM25:", [r["id"] for r in ranked[:2]])
print("recall@2 binario (Cuaderno23):", recallAtK_binary(caso["gold_docs"], ranked, 2))
print("recall@2 real     (corregido):", round(recallAtK(caso["gold_docs"], ranked, 2), 3))


Pregunta: cuándo conviene combinar coincidencia exacta de palabras con parecido conceptual
Gold docs: {'doc_embeddings_01', 'doc_hybrid_01'}
Top-2 BM25: ['doc_embeddings_01', 'doc_vectordb_01']
recall@2 binario (Cuaderno23): 1.0
recall@2 real     (corregido): 0.5


## 9. Aporte agregado: nDCG

El Cuaderno23 **no** implementa nDCG. Lo construyo desde cero con relevancia binaria:

$$\text{DCG@}k=\sum_{i=1}^{k}\frac{rel_i}{\log_2(i+1)},\quad
\text{IDCG@}k=\sum_{i=1}^{\min(k,|G_q|)}\frac{1}{\log_2(i+1)},\quad
\text{nDCG@}k=\frac{\text{DCG@}k}{\text{IDCG@}k}$$

A diferencia de `recall` (cuántos) y `MRR` (el primero), nDCG mide **la calidad del orden
completo**: premia tener los relevantes arriba.

In [11]:
# [AGREGADO por el estudiante]
def dcgAtK(gold_docs: set, retrieved_docs: List[Dict[str, Any]], k: int) -> float:
    # Ganancia acumulada descontada con relevancia binaria.
    dcg = 0.0
    for i, item in enumerate(retrieved_docs[:k], start=1):
        rel = 1.0 if item["id"] in gold_docs else 0.0
        dcg += rel / math.log2(i + 1)
    return dcg


def idcgAtK(gold_docs: set, k: int) -> float:
    # DCG del ranking IDEAL: todos los relevantes primero.
    ideal_hits = min(len(gold_docs), k)
    return sum(1.0 / math.log2(i + 1) for i in range(1, ideal_hits + 1))


def ndcgAtK(gold_docs: set, retrieved_docs: List[Dict[str, Any]], k: int) -> float:
    idcg = idcgAtK(gold_docs, k)
    if idcg == 0.0:
        return 0.0
    return dcgAtK(gold_docs, retrieved_docs, k) / idcg


### 9.1 Validación de nDCG (caso de juguete calculado a mano)

Antes de confiar en la métrica, la verifico contra valores calculados manualmente. Un error en el
descuento o el IDCG daría números plausibles pero incorrectos (error silencioso): por eso valido.

In [12]:
# [AGREGADO por el estudiante]
def fake(ids):  # ranking ficticio a partir de una lista de ids
    return [{"id": x} for x in ids]

# Caso 1: 1 relevante (A) en la posición 2 de [X, A, Y].
#   DCG = 1/log2(3) = 0.63093 ; IDCG = 1/log2(2) = 1 ; nDCG = 0.63093
v1 = ndcgAtK({"A"}, fake(["X", "A", "Y"]), 3)
esperado_1 = 1 / math.log2(3)

# Caso 2: 2 relevantes (A,B) en [A, X, B].
#   DCG = 1/log2(2) + 1/log2(4) = 1 + 0.5 = 1.5
#   IDCG = 1 + 1/log2(3) = 1.63093 ; nDCG = 0.91972
v2 = ndcgAtK({"A", "B"}, fake(["A", "X", "B"]), 3)
esperado_2 = 1.5 / (1 + 1 / math.log2(3))

# Caso 3: ranking ideal [A, B] -> nDCG = 1.0
v3 = ndcgAtK({"A", "B"}, fake(["A", "B"]), 3)

print(f"Caso 1: nDCG={v1:.5f}  esperado={esperado_1:.5f}  ok={abs(v1-esperado_1)<1e-9}")
print(f"Caso 2: nDCG={v2:.5f}  esperado={esperado_2:.5f}  ok={abs(v2-esperado_2)<1e-9}")
print(f"Caso 3 (ideal): nDCG={v3:.5f}  esperado=1.00000  ok={abs(v3-1.0)<1e-9}")
assert abs(v1 - esperado_1) < 1e-9 and abs(v2 - esperado_2) < 1e-9 and abs(v3 - 1.0) < 1e-9
print("\nnDCG validado correctamente.")


Caso 1: nDCG=0.63093  esperado=0.63093  ok=True
Caso 2: nDCG=0.91972  esperado=0.91972  ok=True
Caso 3 (ideal): nDCG=1.00000  esperado=1.00000  ok=True

nDCG validado correctamente.


## 10. Harness de evaluación centrado en retrieval

Adapto el patrón `evaluateCase` / `runBenchmark` / `aggregateMetrics` del Cuaderno23, pero **lo
recorto al retrieval puro** (el Proyecto 6 evalúa recuperación, no generación). El harness recibe
una **función de retrieval** y la evalúa sobre todo el benchmark.

Diseño clave: el retriever devuelve el **ranking completo** y las métricas se calculan recortando
a `k`. Así un mismo ranking sirve para evaluar varios `k` sin volver a recuperar.

In [13]:
# [MODIFICADO de Cuaderno23] (patrón evaluateCase + runBenchmark, recortado a retrieval)
def evaluateRetrievalCase(item: Dict[str, Any], ranking: List[Dict[str, Any]],
                          k: int = 3, k_ndcg: int = 5) -> Dict[str, Any]:
    gold = item["gold_docs"]
    return {
        "question": item["question"],
        "precision_at_k": precisionAtK(gold, ranking, k),
        "recall_at_k": recallAtK(gold, ranking, k),
        "mrr": reciprocalRank(gold, ranking, k),
        "ndcg_at_k": ndcgAtK(gold, ranking, k_ndcg),
        "retrieved_ids": [d["id"] for d in ranking[:k]],
        "gold_docs": gold,
    }


def runRetrievalBenchmark(retrieve_fn: Callable, k: int = 3, k_ndcg: int = 5,
                          **kwargs) -> List[Dict[str, Any]]:
    rows = []
    for item in benchmark:
        ranking = retrieve_fn(item["question"], documents, k=len(documents), **kwargs)
        rows.append(evaluateRetrievalCase(item, ranking, k=k, k_ndcg=k_ndcg))
    return rows


def aggregate(rows: List[Dict[str, Any]]) -> Dict[str, float]:
    metrics = ["precision_at_k", "recall_at_k", "mrr", "ndcg_at_k"]
    return {m: sum(r[m] for r in rows) / len(rows) for m in metrics}


print("Harness de evaluación listo.")


Harness de evaluación listo.


## 11. Línea base: retriever BM25

La **línea base** es BM25 puro (recuperación léxica). Muestro la **evidencia interna**: por consulta, los documentos recuperados con su score, y las métricas agregadas
(`Precision@3`, `Recall@3`, `MRR`, `nDCG@5`).

Uso `k=3` y `nDCG@5` (no `@5`/`@10`) porque el corpus tiene 12 documentos: con `k` grande, el
recall se saturaría a 1 y la comparación perdería poder discriminativo.

In [23]:
# [AGREGADO por el estudiante] usando funciones reutilizadas
K = 3
K_NDCG = 5

baseline_rows = runRetrievalBenchmark(bm25Retrieve, k=K, k_ndcg=K_NDCG)

# Evidencia interna por consulta (formato sugerido)
for item, row in zip(benchmark, baseline_rows):
    ranking = bm25Retrieve(item["question"], documents, k=K)
    print("Query:", item["question"])
    print("Gold:", item["gold_docs"])
    for i, d in enumerate(ranking, start=1):
        marca = "  <-- relevante" if d["id"] in item["gold_docs"] else ""
        print(f"  Doc {i} (score={d['score']:.4f}): {d['id']}{marca}")
    print(f"  P@{K}={row['precision_at_k']:.3f} | R@{K}={row['recall_at_k']:.3f} | "
          f"MRR={row['mrr']:.3f} | nDCG@{K_NDCG}={row['ndcg_at_k']:.3f}")
    print()

base = aggregate(baseline_rows)
print("=== Métricas de retrieval (línea base BM25) ===")
print(f"Precision@{K}: {base['precision_at_k']:.4f}")
print(f"Recall@{K}:    {base['recall_at_k']:.4f}")
print(f"MRR:          {base['mrr']:.4f}")
print(f"nDCG@{K_NDCG}:    {base['ndcg_at_k']:.4f}")


Query: cómo se conecta un modelo de lenguaje con conocimiento externo para responder con respaldo
Gold: {'doc_rag_01'}
  Doc 1 (score=8.9074): doc_prompt_01
  Doc 2 (score=8.3813): doc_rag_01  <-- relevante
  Doc 3 (score=7.9814): doc_vectordb_01
  P@3=0.333 | R@3=1.000 | MRR=0.500 | nDCG@5=0.631

Query: qué método representa el sentido de un texto como vector para comparar conceptos parecidos
Gold: {'doc_embeddings_01'}
  Doc 1 (score=15.6537): doc_embeddings_01  <-- relevante
  Doc 2 (score=9.7896): doc_eval_01
  Doc 3 (score=7.6550): doc_chunking_01
  P@3=0.333 | R@3=1.000 | MRR=1.000 | nDCG@5=1.000

Query: cuándo conviene combinar coincidencia exacta de palabras con parecido conceptual
Gold: {'doc_embeddings_01', 'doc_hybrid_01'}
  Doc 1 (score=7.1338): doc_embeddings_01  <-- relevante
  Doc 2 (score=3.9723): doc_vectordb_01
  Doc 3 (score=3.5194): doc_eval_01
  P@3=0.333 | R@3=0.500 | MRR=1.000 | nDCG@5=0.613

Query: cómo reordenar resultados para poner lo más útil al inicio
Gold:

## 12. Modificación 1 — Ejercicio A: variar `top_k`

**Cambio:** barrer `top_k ∈ {1, 2, 3, 5, 8, 12}` sobre la línea base BM25 y observar P@k, R@k y
nDCG@k. **Valor anterior:** evaluábamos en un único `k=3`.


In [21]:
# [MODIFICADO de Cuaderno23] (celda "Análisis de top-k", ampliada con nDCG y recall real)
def evaluateTopK(retrieve_fn: Callable, k_values: List[int]) -> List[Dict[str, Any]]:
    out = []
    rankings = [retrieve_fn(it["question"], documents, k=len(documents)) for it in benchmark]
    for k in k_values:
        rows = [evaluateRetrievalCase(it, r, k=k, k_ndcg=k) for it, r in zip(benchmark, rankings)]
        agg = aggregate(rows)
        out.append({"top_k": k, "P@k": agg["precision_at_k"],
                    "R@k": agg["recall_at_k"], "nDCG@k": agg["ndcg_at_k"]})
    return out

topk = evaluateTopK(bm25Retrieve, [1,2,3,4,5,8,12])
show_table(["top_k","P@k", "R@k", "nDCG@k"],
          [[t["top_k"], f"{t['P@k']:.3f}", f"{t['R@k']:.3f}", f"{t['nDCG@k']:.3f}"] for t in topk])


# Mini-gráfico ASCII de la tensión precisión vs recall (sin matplotlib).
print("\nTensión precisión (P) vs recall (R) al crecer top_k:")
for t in topk:
    print(f"k={t['top_k']:>2}  P|{'#' * round(t['P@k'] * 30):<30}| {t['P@k']:.2f}"
          f"   R|{'=' * round(t['R@k'] * 30):<30}| {t['R@k']:.2f}")




top_k | P@k   | R@k   | nDCG@k
------------------------------
1     | 0.833 | 0.667 | 0.833 
2     | 0.500 | 0.833 | 0.810 
3     | 0.333 | 0.833 | 0.810 
4     | 0.250 | 0.833 | 0.810 
5     | 0.200 | 0.833 | 0.810 
8     | 0.167 | 1.000 | 0.878 
12    | 0.111 | 1.000 | 0.878 

Tensión precisión (P) vs recall (R) al crecer top_k:
k= 1  P|#########################     | 0.83   R|====================          | 0.67
k= 2  P|###############               | 0.50   R|=========================     | 0.83
k= 3  P|##########                    | 0.33   R|=========================     | 0.83
k= 4  P|########                      | 0.25   R|=========================     | 0.83
k= 5  P|######                        | 0.20   R|=========================     | 0.83
k= 8  P|#####                         | 0.17   R|==============================| 1.00
k=12  P|###                           | 0.11   R|==============================| 1.00


## 13. Modificación 2 — Ejercicio B: comparar configuraciones de retrieval

**Cambio:** comparar la **línea base BM25** contra una **variante con expansión de consulta**
(*query expansion / multi-query*), e incluir denso e híbrido como referencia.

La variante reutiliza la idea de `rewriteQuery` / `multiQueryRetrieve` del Cuaderno23: ante una
consulta parafraseada, **inyecto sinónimos del dominio** para acercar el vocabulario de la consulta
al de los documentos relevantes, y luego recupero con BM25 sobre la consulta expandida.

**Variable que cambio:** el método de recuperación. **Variables constantes:** corpus, benchmark,
`gold_docs`, `k=3`, `k_ndcg=5` y la semilla. Así, cualquier diferencia se debe al retriever.


In [24]:
# [MODIFICADO de Cuaderno23] (variante de expansión basada en rewriteQuery/multiQueryRetrieve)
# Mapa de sinónimos del dominio: acerca el vocabulario de la consulta al de los documentos.
SYNONYM_MAP = {
    "modelo de lenguaje": "llm",
    "conocimiento externo": "documentos externos evidencia",
    "respaldo": "evidencia fundamentada soporte documental",
    "sentido": "significado",
    "fundamentada": "evidencia soporte documental grounded",
    "conceptos parecidos": "similitud semántica conceptual",
    "parecido conceptual": "similitud semántica",
    "coincidencia exacta de palabras": "bm25 términos exactos",
    "reordenar resultados": "reranking reordena documentos recuperados",
    "lo más útil al inicio": "evidencia útil primeras posiciones",
    "mecanismos": "guardrails controles",
    "programa": "agente",
    "acciones y observar": "herramientas observa acción observación",
}


def expandQuery(question: str) -> str:
    # Expande la consulta inyectando sinónimos del dominio que aparezcan en ella.
    nq = normalizeText(question)
    extra = [expansion for key, expansion in SYNONYM_MAP.items() if normalizeText(key) in nq]
    return question + " " + " ".join(extra) if extra else question


def bm25ExpandedRetrieve(query: str, docs: List[Document], k: int = 3) -> List[Dict[str, Any]]:
    # Variante: recupera con BM25 sobre la consulta expandida.
    return bm25Retrieve(expandQuery(query), docs, k=k)


# Ejemplo: cómo cambia una consulta tras la expansión.
ej = benchmark[0]["question"]
print("Original :", ej)
print("Expandida:", expandQuery(ej))


Original : cómo se conecta un modelo de lenguaje con conocimiento externo para responder con respaldo
Expandida: cómo se conecta un modelo de lenguaje con conocimiento externo para responder con respaldo llm documentos externos evidencia evidencia fundamentada soporte documental


In [26]:
# [AGREGADO por el estudiante]
configs = {
    "BM25 (base)":     lambda q, d, k: bm25Retrieve(q, d, k=k),
    "Denso TF-IDF":    lambda q, d, k: denseRetrieve(q, d, k=k),
    "Híbrido a=0.6":   lambda q, d, k: hybridRetrieve(q, d, k=k, alpha=0.6),
    "BM25+expansión":  lambda q, d, k: bm25ExpandedRetrieve(q, d, k=k),
}

resultados = {}
for name, fn in configs.items():
    rows = runRetrievalBenchmark(fn, k=K, k_ndcg=K_NDCG)
    resultados[name] = (rows, aggregate(rows))

show_table(
    ["Configuración", f"P@{K}", f"R@{K}", "MRR", f"nDCG@{K_NDCG}"],
    [[name, f"{a['precision_at_k']:.3f}", f"{a['recall_at_k']:.3f}",
      f"{a['mrr']:.3f}", f"{a['ndcg_at_k']:.3f}"] for name, (_, a) in resultados.items()],
)

# Desglose por consulta de recall@3: dónde gana la expansión.
print("\nRecall@3 por consulta (BM25 vs BM25+expansión):")
for i, item in enumerate(benchmark):
    rb = resultados["BM25 (base)"][0][i]["recall_at_k"]
    re_ = resultados["BM25+expansión"][0][i]["recall_at_k"]
    flag = "  <- la expansión mejora" if re_ > rb else ("  <- empeora" if re_ < rb else "")
    print(f"  R@3 BM25={rb:.2f} | +exp={re_:.2f} | {item['question'][:55]}{flag}")


Configuración  | P@3   | R@3   | MRR   | nDCG@5
-----------------------------------------------
BM25 (base)    | 0.333 | 0.833 | 0.917 | 0.810 
Denso TF-IDF   | 0.333 | 0.833 | 0.917 | 0.810 
Híbrido a=0.6  | 0.333 | 0.833 | 0.917 | 0.810 
BM25+expansión | 0.444 | 1.000 | 1.000 | 1.000 

Recall@3 por consulta (BM25 vs BM25+expansión):
  R@3 BM25=1.00 | +exp=1.00 | cómo se conecta un modelo de lenguaje con conocimiento 
  R@3 BM25=1.00 | +exp=1.00 | qué método representa el sentido de un texto como vecto
  R@3 BM25=0.50 | +exp=1.00 | cuándo conviene combinar coincidencia exacta de palabra  <- la expansión mejora
  R@3 BM25=1.00 | +exp=1.00 | cómo reordenar resultados para poner lo más útil al ini
  R@3 BM25=1.00 | +exp=1.00 | qué mecanismos controlan permisos, formato y salidas de
  R@3 BM25=0.50 | +exp=1.00 | qué arquitectura permite a un programa decidir usar acc  <- la expansión mejora


## 14. Comparación final: línea base vs variante 

Resumen cuantitativo BM25 (A) vs BM25+expansión (B), en el formato de salida del enunciado.

In [27]:
# [AGREGADO por el estudiante]
a = resultados["BM25 (base)"][1]
b = resultados["BM25+expansión"][1]

print("Comparación configuración A (BM25) vs. B (BM25+expansión):")
print(f"A: P@{K}={a['precision_at_k']:.4f}, R@{K}={a['recall_at_k']:.4f}, "
      f"MRR={a['mrr']:.4f}, nDCG@{K_NDCG}={a['ndcg_at_k']:.4f}")
print(f"B: P@{K}={b['precision_at_k']:.4f}, R@{K}={b['recall_at_k']:.4f}, "
      f"MRR={b['mrr']:.4f}, nDCG@{K_NDCG}={b['ndcg_at_k']:.4f}")

print("\nDelta (B - A):")
for m, label in [("precision_at_k", f"P@{K}"), ("recall_at_k", f"R@{K}"),
                 ("mrr", "MRR"), ("ndcg_at_k", f"nDCG@{K_NDCG}")]:
    d = b[m] - a[m]
    print(f"  {label:10s}: {'+' if d >= 0 else ''}{d:.4f}")


Comparación configuración A (BM25) vs. B (BM25+expansión):
A: P@3=0.3333, R@3=0.8333, MRR=0.9167, nDCG@5=0.8095
B: P@3=0.4444, R@3=1.0000, MRR=1.0000, nDCG@5=1.0000

Delta (B - A):
  P@3       : +0.1111
  R@3       : +0.1667
  MRR       : +0.0833
  nDCG@5    : +0.1905


## 15. Análisis de errores

Adapto la idea de `classifyFailure` del Cuaderno23 a un diagnóstico **de retrieval**: clasifico
cada consulta de la línea base según su patrón de fallo dominante.

In [19]:
# [MODIFICADO de Cuaderno23] (celda "Diagnóstico de fallos", adaptada a retrieval)
def classifyRetrievalFailure(row: Dict[str, Any]) -> str:
    if row["recall_at_k"] == 0.0:
        return "retrieval pobre (ningún gold en top_k)"
    if row["recall_at_k"] < 1.0:
        return "recall parcial (falta algún gold)"
    if row["precision_at_k"] < 0.34:
        return "contexto ruidoso (mucho irrelevante en top_k)"
    if row["mrr"] < 1.0:
        return "primer relevante no está en la posición 1"
    return "sin fallo crítico"


print("Diagnóstico por consulta (línea base BM25, k=3):")
for item, row in zip(benchmark, baseline_rows):
    print(f"  [{classifyRetrievalFailure(row)}] {item['question'][:60]}")


Diagnóstico por consulta (línea base BM25, k=3):
  [contexto ruidoso (mucho irrelevante en top_k)] cómo se conecta un modelo de lenguaje con conocimiento exter
  [contexto ruidoso (mucho irrelevante en top_k)] qué método representa el sentido de un texto como vector par
  [recall parcial (falta algún gold)] cuándo conviene combinar coincidencia exacta de palabras con
  [contexto ruidoso (mucho irrelevante en top_k)] cómo reordenar resultados para poner lo más útil al inicio
  [contexto ruidoso (mucho irrelevante en top_k)] qué mecanismos controlan permisos, formato y salidas de un s
  [recall parcial (falta algún gold)] qué arquitectura permite a un programa decidir usar acciones


## 16. Respuestas a las preguntas avanzadas obligatorias

**1. ¿Por qué precisión y recall están en tensión y cómo se relacionan con `top_k`?**
Al aumentar `top_k` el recall no puede bajar (entran más candidatos, se recuperan más relevantes),
pero la precisión tiende a caer porque, habiendo pocos relevantes por consulta, las posiciones
extra se llenan de distractores. Son objetivos opuestos: maximizar uno suele degradar el otro. El
barrido de la Modificación 1 muestra exactamente ese cruce.

**2. ¿Qué limitación tiene MRR cuando hay múltiples documentos relevantes?**
MRR solo mira la posición del **primer** acierto: una vez encontrado uno, ignora si los demás
relevantes están en el puesto 2 o nunca aparecen. En nuestras dos consultas con 2 gold docs, un
sistema que pone un relevante primero y olvida el segundo obtiene MRR perfecto pero recall y nDCG
imperfectos. Por eso reporto las cuatro métricas juntas.

**3. ¿Por qué nDCG es más informativo que precision para rankings ordenados?**
Precision@k trata todas las posiciones del top-$k$ por igual (conteo sin orden). nDCG aplica un
**descuento logarítmico**: un relevante en la posición 1 vale más que en la 5, y normaliza por el
ranking ideal (IDCG). Así captura **calidad de ordenamiento**, no solo presencia.

**4. ¿Cómo se relaciona la calidad métrica de retrieval con la calidad de la respuesta generada?**
El retrieval es el techo de un RAG: si los relevantes no entran al contexto (recall bajo) o entran
abajo y se truncan (nDCG bajo), el generador no tiene la evidencia y alucina o responde incompleto.
Mejor retrieval ⇒ mejor techo para la generación fundamentada (aunque no lo garantiza).

**5. ¿Qué sesgo introduce construir tu propio dataset de evaluación?**
Yo definí consultas y `gold_docs`, así que el benchmark refleja mi interpretación de "relevante"
(sesgo del anotador). Es pequeño (6 consultas, 12 documentos) y de baja cobertura. Para no
**favorecer artificialmente a BM25** parafraseé las consultas (desajuste de vocabulario); aun así,
elegir qué sinónimos inyecta la expansión es otra decisión mía que puede inflar su ventaja. Las
cifras absolutas no son generalizables; solo son válidas para comparar configuraciones **bajo las
mismas condiciones**.


## 17. Respuestas a preguntas transversales

**¿Qué parte de tu trabajo es retrieval, generación y razonamiento del agente?**
Todo es **retrieval y su evaluación**; no hay generación ni agente (el Proyecto 6 se centra en
recuperación). El "razonamiento" es el análisis comparativo de métricas.

**¿Qué componente sería más difícil de detectar si estuviera mal implementado?**
nDCG: un error en el descuento o en el IDCG produce números plausibles (en $[0,1]$) pero
incorrectos. Por eso añadí la **celda de validación** con casos calculados a mano.

**¿Qué resultado podría parecer bueno pero ser técnicamente engañoso?**
Un `recall@k` alto usando la versión **binaria** original del Cuaderno23: marca éxito aunque falte
la mitad de los relevantes. La corrección a recall real lo desenmascara.

**¿Qué variable cambiaste y cuál mantuviste constante para que la comparación sea justa?**
Cambié el método de recuperación (BM25 vs denso vs híbrido vs expansión). Mantuve constantes
corpus, benchmark, `gold_docs`, `k`, `k_ndcg` y la semilla.

**¿Qué parte de tu resultado depende del corpus y no del modelo?**
La ventaja de la expansión depende del **mapa de sinónimos** y de que las consultas estén
parafraseadas: es una propiedad del corpus/consultas, no un algoritmo universalmente superior.

**¿Qué parte de tu código permite reproducibilidad?**
`random.seed(42)` y el carácter **determinista** del pipeline (sin muestreo): dos ejecuciones dan
resultados idénticos. La celda de verificación personal documenta entorno y semilla.

**¿Dónde aparece la función de similitud o métrica de evaluación?**
La similitud está en `cosineSimilarity` (denso) y en el score BM25; las métricas en `precisionAtK`,
`recallAtK`, `reciprocalRank` y `ndcgAtK`.


## 18. Conclusión técnica

Implementé un evaluador de retrieval reproducible sobre un corpus enriquecido (12 documentos, con
distractores) y un benchmark de consultas parafraseadas. Agregué **nDCG** (ausente en el cuaderno
base), **corregí** `recall@k` para que mida cobertura real, y comparé una línea base **BM25** contra
varias variantes con cuatro métricas, además de barrer `top_k`. Los experimentos evidencian la
**tensión precisión–recall** y muestran que, ante desajuste de vocabulario, las variantes léxicas
(denso/híbrido) aportan poco mientras que la **expansión de consulta** mejora claramente el
retrieval. Las conclusiones son **relativas** (comparación bajo condiciones idénticas), no
absolutas, dadas las limitaciones del dataset.

## 19. Declaración de autoría y uso de IA

> Declaro que comprendo el código, los resultados y las explicaciones entregadas en esta Práctica.
> Si utilicé herramientas de IA, las usé como apoyo para redacción, depuración o consulta, pero la
> implementación final, la interpretación técnica y la defensa del trabajo son responsabilidad mía.

**Uso concreto de IA en este trabajo:**
- Usé IA para revisar la redacción del README y de las explicaciones teóricas.
- Usé IA para contrastar mi fórmula de nDCG (DCG/IDCG) y diseñar los casos de validación a mano.
- La implementación de `ndcgAtK`, la corrección de `recallAtK`, el diseño del corpus/benchmark, la
  variante de expansión y todos los análisis e interpretaciones son míos.

**Trazabilidad:** cada celda de código indica si fue reutilizada, modificada o agregada (ver la
convención al inicio del notebook).

## 20. Qué hice, por qué lo hice y qué significan mis resultados

**Qué hice:** un sistema de evaluación de retrieval que reutiliza recuperadores y métricas base del
Cuaderno23, al que **agregué nDCG**, **corregí el recall**, **enriquecí el corpus/benchmark** y
**comparé** BM25 contra una variante con expansión de consulta, barriendo `top_k`.

**Por qué lo hice:** elegir un retriever "a ojo" no es defendible. Sin nDCG faltaba la única
métrica sensible al **orden**; el `recall` original sobreestimaba el éxito con varios relevantes; y
sin distractores ni paráfrasis el problema era trivial. Corregir y completar la evaluación la hace
honesta y discriminativa.

**Qué significan los resultados:** las métricas cuantifican que (a) existe una tensión real entre
precisión y recall gobernada por `top_k`, y (b) cuando el problema es de **vocabulario**, mejorar
el algoritmo de puntuación rinde poco frente a **mejorar la consulta**. La mejora de la expansión
es **medible**, no cosmética, y tiene una explicación mecánica clara.

## 21. Puente al curso

Este proyecto conecta con varios temas del curso CC0C2:

- **Evaluación de retrieval (Cuaderno 23):** núcleo del trabajo; extiendo sus métricas con nDCG y
  un recall corregido.
- **Búsqueda semántica y embeddings (Cuaderno 21):** el denso TF-IDF simula la similitud semántica;
  su limitación (sigue siendo léxico) motiva, en un sistema real, sustituirlo por embeddings densos
  (`sentence-transformers`) conservando el mismo harness de evaluación.
- **Arquitectura RAG (Cuaderno 22):** el retrieval es la primera etapa de un pipeline RAG; estas
  métricas acotan el techo de la generación fundamentada.
- **Grounded generation (Cuaderno 23):** un retrieval con buen recall/nDCG es condición necesaria
  (no suficiente) para respuestas fieles al contexto; sin evidencia recuperada, el LLM alucina.
